In [ ]:
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PDFS_PATH = str(REPO_ROOT / "data" / "mcq" / "pdf")
OUTPUT_FILE = "./questions.json"
MIN_IMAGE_BYTES = 5000
PAGE_RENDER_DPI = 150

In [ ]:
# Qwen2.5-VL page parser: reads one rendered PDF page image directly and emits
# structured question(s) found on that page in a single pass. Same
# model/loading pattern as src/captioning/qwen_vl.py (QwenVLCaptioner) —
# lazy load/unload, one model resident at a time on the unified-memory budget,
# bfloat16, load-to-CPU-then-move (MPS fp16 cast crashes on direct device_map
# load, see qwen_vl.py docstring).
#
# 3B instead of 7B: same architecture/prompting, smaller weights -> faster
# generate() per page on MPS (no flash-attention there, so wall-clock is
# dominated by raw compute) at some accuracy cost on harder pages — worth
# the trade for this batch-extraction workload.
#
# This replaces the fitz line/bold/regex layout heuristics entirely: the
# model sees the page as an image and does segmentation + stem/option/answer/
# reference extraction itself, instead of us reconstructing layout from text
# spans first and only handing the LLM an already-segmented block.

import gc
import os
import time

os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

import torch
from PIL import Image
from qwen_vl_utils import process_vision_info
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

MODEL_PATH = "Qwen/Qwen2.5-VL-3B-Instruct"


def get_device() -> str:
    if torch.cuda.is_available():
        return "cuda"
    if torch.backends.mps.is_available():
        return "mps"
    return "cpu"


class PageParser:
    def __init__(self, model_path: str = MODEL_PATH):
        self.model_path = model_path
        self.device = get_device()
        self.model = None
        self.processor = None

    def load(self):
        if self.model is not None:
            print("Model already loaded, skipping.")
            return
        print(f"Loading processor for {self.model_path}...")
        self.processor = AutoProcessor.from_pretrained(self.model_path)
        print(f"Loading model weights for {self.model_path} (this can take a while on first download)...")
        t0 = time.monotonic()
        self.model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
            self.model_path,
            torch_dtype=torch.bfloat16,
        )
        print(f"Weights loaded in {time.monotonic() - t0:.1f}s, moving to {self.device}...")
        self.model.to(self.device)
        self.model.eval()
        print("Model ready.")

    def unload(self):
        self.model = None
        self.processor = None
        gc.collect()
        if torch.backends.mps.is_available():
            torch.mps.empty_cache()
        elif torch.cuda.is_available():
            torch.cuda.empty_cache()

    def generate(
        self,
        image: Image.Image,
        prompt: str,
        max_new_tokens: int = 1024,
        max_pixels: int = 1_280_000,
    ) -> str:
        """max_pixels caps the page render before tokenization — see
        qwen_vl.py's transcribe_page docstring on why an uncapped full-page
        image burns thousands of vision tokens on prefill, and why this is
        especially costly on MPS (no flash-attention / optimized kernels
        there — generation is materially slower than on CUDA). max_new_tokens
        is capped well below the JSON's realistic worst case (a couple of
        questions per page) so a slow/looping generation doesn't run all the
        way to a much larger ceiling before returning."""
        assert self.model is not None, "call load() first"
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image.convert("RGB"), "max_pixels": max_pixels},
                    {"type": "text", "text": prompt},
                ],
            }
        ]
        text = self.processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        image_inputs, video_inputs = process_vision_info(messages)
        inputs = self.processor(
            text=[text],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt",
        ).to(self.device)
        if "pixel_values" in inputs:
            inputs["pixel_values"] = inputs["pixel_values"].to(torch.bfloat16)

        with torch.inference_mode():
            output_ids = self.model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)

        new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
        return self.processor.decode(new_tokens, skip_special_tokens=True)


page_parser = PageParser()
page_parser.load()
print(f"Loaded {MODEL_PATH} on {page_parser.device}")
if page_parser.device != "mps" and torch.backends.mps.is_available():
    print("WARNING: MPS is available but model is not using it — check get_device() / device placement.")

In [ ]:
import io
import json
import re
import time

import fitz
from PIL import Image
from tqdm.auto import tqdm

# ── image crop extraction (kept: questions still need has_image/image_paths
# for embedded figures, independent of the page-level vision parse) ───────
def save_images(doc, out_dir: Path, stem: str) -> list:
    saved, seen = [], set()
    (out_dir / "images").mkdir(parents=True, exist_ok=True)
    for pnum, page in enumerate(doc):
        for img in page.get_images(full=True):
            xref = img[0]
            if xref in seen: continue
            seen.add(xref)
            try:
                bi = doc.extract_image(xref)
                if not bi or bi["size"] < MIN_IMAGE_BYTES: continue
                fname = f"{stem}_p{pnum+1}_x{xref}.{bi['ext']}"
                fpath = out_dir / "images" / fname
                fpath.write_bytes(bi["image"])
                saved.append({"page": pnum, "path": f"images/{fname}"})
            except Exception:
                pass
    return saved


def render_page(page, dpi: int = PAGE_RENDER_DPI) -> Image.Image:
    pix = page.get_pixmap(dpi=dpi)
    return Image.open(io.BytesIO(pix.tobytes("png")))


def extract_page_text(page) -> str:
    """Pulls the PDF's native text layer for this page (these are
    text-based PDFs, not scans, so fitz's own extraction works as OCR
    grounding — no separate OCR engine needed). Passed to the vision LLM
    as ground-truth text so it transcribes/copies known characters instead
    of re-reading them from pixels, same role as the MinerU OCR `labels`
    passed into QwenVLCaptioner.caption() in src/captioning/qwen_vl.py."""
    return page.get_text().strip()


# ── LLM-based page parsing ─────────────────────────────────────────────────
# One vision-LLM call per rendered page image replaces fitz's line/bold
# reconstruction, the numbered-block regex segmenter, and the format
# detector: the model reads the page like a person would and returns
# already-segmented, already-structured questions directly. The page's OCR
# text layer is passed alongside the image as grounding context (see
# extract_page_text) so the model copies known text instead of re-reading
# every character from pixels — faster and more accurate transcription,
# while the image still carries the layout/bold/circled-answer signal text
# alone can't give us.

PAGE_PARSE_PROMPT = """This image is one page from an Indonesian medical multiple-choice exam PDF. Find every question on this page and extract it.

Return ONLY a JSON array, no markdown fences, no commentary. Each element has this exact shape:
{
  "number": <the question's printed number, or null if this is a continuation (see below)>,
  "background": "the clinical vignette/scenario text (patient history, exam findings, case setup), or empty string if this question has no separate scenario",
  "question": "the actual question being asked — the sentence that ends with what to answer (often starts with 'Apakah', 'Bagaimana', 'Manakah', 'Apa')",
  "options": {"A": "...", "B": "...", ...},
  "answer": "A" | null,
  "reference": "citation/source text if present (e.g. 'Referensi: ...' book/journal/author/page), else empty string",
  "has_image": true | false
}

Rules:
- "answer" is the correct option letter ONLY if explicitly marked on the page (bold text, circled/highlighted option, or an explicit "ANSWER: X" / "Jawaban: X" line). Use null if no marker is present — do not guess from medical knowledge.
- Options must be transcribed verbatim, just trimmed.
- Split the vignette from the ask: "background" is the case setup (patient demographics, history, exam/lab findings) — usually starts with "Seorang", "Seseorang", "Pasien", "Pada ...". "question" is just the final question sentence itself (e.g. "Apakah diagnosis yang paling mungkin?"), not the case details. Many questions have no real background (a bare factual/conceptual question) — in that case set "background" to "" and put the whole question text in "question". Keep both separate from any reference/citation text even if adjacent on the page.
- "has_image" is true if the question is illustrated by a photo, diagram, chart, or scan on this page (not counting decorative page headers/logos).
- If the FIRST question on the page is a continuation of a question whose background/question/options started on the PREVIOUS page (e.g. this page starts mid-option-list with no background/question text, or starts with "Referensi:" / an "ANSWER:" line with no preceding text), set "number" to null for that entry and put whatever text appears on THIS page into its fields — it will be merged with the previous page's last question.
- Ignore page numbers, "Paket X" headers/footers, and running headers unrelated to question content.
- If the page has no questions at all (e.g. a cover page), return an empty array []."""

OCR_CONTEXT_TEMPLATE = """

The following is the raw text layer extracted from this same page (reading order may be jumbled by column/box layout — use the IMAGE to determine correct reading order and to see bold/circled/highlighted markers, which this raw text does not preserve). Use it to transcribe exact characters/spelling instead of reading them off the image:

{ocr_text}"""

_THINK_BLOCK_RE = re.compile(r"<think>.*?</think>", re.DOTALL)
_JSON_ARRAY_RE = re.compile(r"\[.*\]", re.DOTALL)


def _repair_truncated_json_array(raw: str) -> str:
    """Best-effort fix for a JSON array cut off mid-generation
    (max_new_tokens truncation): drops the last, possibly-incomplete
    element and closes the array, so json.loads can recover every fully
    formed question that came before it."""
    text = raw[raw.index("["):] if "[" in raw else raw

    if text.count('"') % 2 == 1:
        text = text[: text.rindex('"')]

    depth = 0
    last_complete_end = None
    in_string = False
    escape = False
    for i, ch in enumerate(text):
        if in_string:
            if escape:
                escape = False
            elif ch == "\\":
                escape = True
            elif ch == '"':
                in_string = False
            continue
        if ch == '"':
            in_string = True
        elif ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                last_complete_end = i

    if last_complete_end is None:
        raise ValueError("no complete element found")
    return text[: last_complete_end + 1] + "]"


def _call_llm_json_array(image: Image.Image, ocr_text: str, log_prefix: str) -> list:
    prompt = PAGE_PARSE_PROMPT
    if ocr_text:
        prompt += OCR_CONTEXT_TEMPLATE.format(ocr_text=ocr_text)

    tqdm.write(f"{log_prefix} rendering ok, calling LLM (generate)...")
    t0 = time.monotonic()
    raw = page_parser.generate(image, prompt)
    dt = time.monotonic() - t0
    tqdm.write(f"{log_prefix} LLM generate done in {dt:.1f}s, {len(raw)} chars raw output — parsing JSON...")

    cleaned = _THINK_BLOCK_RE.sub("", raw)
    cleaned = re.sub(r"^```(?:json)?\s*|\s*```$", "", cleaned.strip())

    matches = _JSON_ARRAY_RE.findall(cleaned)
    if not matches:
        raise ValueError(f"No JSON array found in LLM output: {raw[:200]!r}")
    candidate = matches[-1]

    try:
        return json.loads(candidate)
    except json.JSONDecodeError:
        tqdm.write(f"{log_prefix} JSON truncated, attempting repair...")
        return json.loads(_repair_truncated_json_array(candidate))


def _normalize_item(item: dict) -> dict:
    options = {str(k).upper(): str(v).strip()
               for k, v in (item.get("options") or {}).items()
               if str(k).upper() in "ABCDE"}
    answer = item.get("answer")
    if answer:
        answer = str(answer).strip().upper()
        if answer not in "ABCDE" or answer not in options:
            answer = None
    return {
        "background": str(item.get("background") or "").strip(),
        "question": str(item.get("question") or "").strip(),
        "options": {k: options[k] for k in sorted(options)},
        "answer": answer,
        "reference": str(item.get("reference") or "").strip(),
        "has_image": bool(item.get("has_image")),
    }


def _merge_continuation(prev: dict, cont: dict) -> None:
    """Folds a continuation entry (number == null) into the previous
    question in place: fills in whatever fields the previous page's parse
    left empty, and appends any extra options/reference/answer found."""
    if not prev["background"] and cont["background"]:
        prev["background"] = cont["background"]
    if not prev["question"] and cont["question"]:
        prev["question"] = cont["question"]
    for k, v in cont["options"].items():
        prev["options"].setdefault(k, v)
    prev["options"] = {k: prev["options"][k] for k in sorted(prev["options"])}
    if not prev["answer"] and cont["answer"]:
        prev["answer"] = cont["answer"]
    if cont["reference"]:
        prev["reference"] = (prev["reference"] + " " + cont["reference"]).strip()
    prev["has_image"] = prev["has_image"] or cont["has_image"]


# ── main function ─────────────────────────────────────────────────────────
def extract_pdf(pdf_path: str, out_dir: Path) -> list:
    file_stem = Path(pdf_path).stem
    tqdm.write(f"\n[{file_stem}] opening PDF...")
    try:
        doc = fitz.open(pdf_path)
    except Exception as e:
        tqdm.write(f"  [skip] {Path(pdf_path).name}: {e}")
        return []

    tqdm.write(f"[{file_stem}] {len(doc)} pages — extracting embedded images...")
    images = save_images(doc, out_dir, file_stem)
    tqdm.write(f"[{file_stem}] {len(images)} embedded image(s) saved")
    images_by_page: dict[int, list] = {}
    for img in images:
        images_by_page.setdefault(img["page"], []).append(img["path"])

    parsed_questions: list[dict] = []
    for pnum in tqdm(range(len(doc)), desc=file_stem, unit="pg", leave=False):
        log_prefix = f"[{file_stem}] page {pnum + 1}/{len(doc)}:"
        tqdm.write(f"{log_prefix} rendering page @ {PAGE_RENDER_DPI} DPI...")
        page_img = render_page(doc[pnum])
        ocr_text = extract_page_text(doc[pnum])
        try:
            items = _call_llm_json_array(page_img, ocr_text, log_prefix)
        except Exception as e:
            tqdm.write(f"  [warn] LLM parse failed for {file_stem} p{pnum+1}: {e}")
            continue

        tqdm.write(f"{log_prefix} {len(items)} item(s) found")
        page_img_paths = images_by_page.get(pnum, [])
        for raw_item in items:
            norm = _normalize_item(raw_item)
            if raw_item.get("number") is None and parsed_questions:
                tqdm.write(f"{log_prefix} merging continuation into previous question")
                _merge_continuation(parsed_questions[-1], norm)
                if page_img_paths:
                    parsed_questions[-1]["image_paths"].extend(page_img_paths)
            else:
                norm["image_paths"] = list(page_img_paths) if norm["has_image"] else []
                parsed_questions.append(norm)

    doc.close()

    questions = []
    for idx, q in enumerate(parsed_questions, 1):
        if not (q["options"] and (q["background"] or q["question"] or q["has_image"])):
            continue
        questions.append({
            "id": f"{file_stem}_Q{idx:03d}",
            "source": f"{file_stem}.pdf",
            "format": "vision",
            "background": q["background"],
            "question": q["question"],
            "options": q["options"],
            "answer": q["answer"],
            "reference": q["reference"],
            "has_image": bool(q["image_paths"]),
            "image_paths": q["image_paths"],
        })
    tqdm.write(f"[{file_stem}] done — {len(questions)} question(s) extracted")
    return questions

print("Extractor defined.")

In [ ]:
import time

out_dir = Path(OUTPUT_FILE).parent
pdf_files = sorted(Path(PDFS_PATH).glob("*.pdf"))

if not pdf_files:
    print(f"No PDFs found in {INPUT_DIR}")
else:
    print(f"Found {len(pdf_files)} PDF files\n")

all_questions = []
run_start = time.monotonic()

for i, pdf in enumerate(tqdm(pdf_files, desc="PDFs", unit="file"), 1):
    tqdm.write(f"\n=== [{i}/{len(pdf_files)}] {pdf.name} ===")
    pdf_start = time.monotonic()
    qs    = extract_pdf(str(pdf), out_dir)
    pdf_dt = time.monotonic() - pdf_start
    auto  = sum(1 for q in qs if q["answer"])
    imgs  = sum(1 for q in qs if q["has_image"])
    fmt   = qs[0]["format"] if qs else "?"
    tqdm.write(f"  {pdf.name:<45}  Fmt:{fmt}  {len(qs):>2}Q  "
               f"{auto:>2} w/answer  {imgs} img  ({pdf_dt:.1f}s)")
    all_questions.extend(qs)

total_dt = time.monotonic() - run_start
print(f"\nAll PDFs processed in {total_dt:.1f}s")

# save
Path(OUTPUT_FILE).write_text(
    json.dumps(all_questions, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

total  = len(all_questions)
w_ans  = sum(1 for q in all_questions if q["answer"])
no_ans = total - w_ans
imgs   = sum(1 for q in all_questions if q["has_image"])

print(f"""
{'='*50}
Total questions  : {total}
With answer      : {w_ans}
Without answer   : {no_ans}  ← fill manually or use LLM
With images      : {imgs}
Output           : {OUTPUT_FILE}
{'='*50}
""")

In [ ]:
for q in all_questions[:3]:
    print(f"[{q['id']}]  fmt:{q['format']}  answer:{q['answer']}")
    if q["background"]:
        print(f"  BG: {q['background'][:100]}...")
    print(f"  Q : {q['question'][:100]}")
    for k, v in q["options"].items():
        marker = " ◀" if k == q["answer"] else ""
        print(f"    {k}. {v[:60]}{marker}")
    print()
